In [1]:
import os
from dotenv import load_dotenv
import gradio as gr
import google.generativeai as genai
from openai import OpenAI
import requests

# =========================
# 🔑 LOAD ENV
# =========================
load_dotenv()

# =========================
# 🤖 CLIENTS
# =========================
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

def get_gemini():
    try:
        return genai.GenerativeModel("gemini-2.5-flash-lite")
    except:
        return genai.GenerativeModel("gemini-1.5-flash")

gemini_model = get_gemini()

# =========================
# 🏆 LEADERBOARD
# =========================
leaderboard = {"Alex": 0, "Blake": 0, "Charlie": 0}

# =========================
# 🧠 MODEL ROUTER
# =========================
def ask_model(role, personality, conversation, model_name):

    prompt = f"""
You are {role}. {personality}

Conversation:
{conversation}

Reply in 2 lines max. Stay in character.
"""

    # OpenAI
    if model_name == "gpt-4o-mini":
        res = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": f"You are {role}. {personality}"},
                {"role": "user", "content": prompt}
            ]
        )
        return res.choices[0].message.content.strip()

    # Gemini
    elif model_name == "gemini":
        res = gemini_model.generate_content(prompt)
        return res.text.strip()

    # Ollama
    elif model_name.startswith("ollama"):
        model = model_name.split(":")[1]
        try:
            res = requests.post(
                "http://localhost:11434/api/generate",
                json={"model": model, "prompt": prompt, "stream": False}
            )
            return res.json()["response"]
        except:
            return "⚠️ Ollama not running"

    return "⚠️ Model not available"


# =========================
# ⚖️ JUDGE
# =========================
def judge(conversation):
    try:
        res = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Pick winner: Alex, Blake, or Charlie. Only name."},
                {"role": "user", "content": conversation}
            ]
        )
        winner = res.choices[0].message.content.strip().replace(".", "")
        if winner in leaderboard:
            leaderboard[winner] += 1
        return winner
    except:
        return "No Judge"


# =========================
# 🔁 MAIN
# =========================
def run(topic, turns, alex_model, blake_model, charlie_model):

    if not topic:
        return "⚠️ Enter a topic!", format_board()

    conversation = f"Topic: {topic}\n"
    chat_output = []

    for _ in range(int(turns)):

        alex = ask_model("Alex", "argumentative and sarcastic", conversation, alex_model)
        conversation += f"\n[Alex]: {alex}"
        chat_output.append(("Alex", alex))

        blake = ask_model("Blake", "calm and logical", conversation, blake_model)
        conversation += f"\n[Blake]: {blake}"
        chat_output.append(("Blake", blake))

        charlie = ask_model("Charlie", "funny and casual", conversation, charlie_model)
        conversation += f"\n[Charlie]: {charlie}"
        chat_output.append(("Charlie", charlie))

        conversation = conversation[-3000:]

    winner = judge(conversation)
    chat_output.append(("🏆 Winner", winner))

    return chat_output, format_board()


# =========================
# 🏆 FORMAT BOARD
# =========================
def format_board():
    return "\n".join([f"{k}: {v}" for k, v in leaderboard.items()])


def reset():
    for k in leaderboard:
        leaderboard[k] = 0
    return format_board()


# =========================
# 🎨 UI (CLEAN STRUCTURE)
# =========================
with gr.Blocks() as demo:

    gr.Markdown("# 🤖 Multi-Agent AI Arena")

    with gr.Row():

        # LEFT PANEL (Controls)
        with gr.Column(scale=1):
            topic = gr.Textbox(label="Topic")
            turns = gr.Slider(1, 5, value=3, label="Rounds")

            gr.Markdown("### 🧠 Model Selection")

            model_options = ["gpt-4o-mini", "gemini", "ollama:llama3"]

            alex_model = gr.Dropdown(model_options, value="gpt-4o-mini", label="Alex")
            blake_model = gr.Dropdown(model_options, value="gemini", label="Blake")
            charlie_model = gr.Dropdown(model_options, value="gpt-4o-mini", label="Charlie")

            run_btn = gr.Button("🚀 Start Debate")
            reset_btn = gr.Button("🔄 Reset Leaderboard")

        # RIGHT PANEL (Output)
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversation")
            leaderboard_box = gr.Textbox(label="🏆 Leaderboard")

    run_btn.click(
        run,
        inputs=[topic, turns, alex_model, blake_model, charlie_model],
        outputs=[chatbot, leaderboard_box]
    )

    reset_btn.click(reset, outputs=leaderboard_box)

# =========================
# 🚀 RUN
# =========================
demo.launch()

C:\Users\AMIT KUMAR BEHERA\AppData\Local\Temp\ipykernel_26664\2849728543.py:171: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
